# ATLAS Wind Scaling

This notebook combines two wind products:

1. the downscaled wind component from the reanalysis workflow;
2. the station based IDW wind component.

The final product is a weighted integration of the two sources. The IDW contribution is stronger close to stations and weaker far from stations.


The integration uses the following weighted average:

$$
\Huge
component_\mathrm{integrated} = \frac{\frac{\mathrm{component}_{\mathrm{era5}}}{\sigma_{\mathrm{era5}}^2} + \frac{\mathrm{component}_{\mathrm{idw}}}{\sigma_{\mathrm{idw}}^2}}{\frac{1}{\sigma_{\mathrm{era5}}^2} + \frac{1}{\sigma_{\mathrm{idw}}^2}}
$$

$\sigma$ is a coefficient that measures the uncertainty of the data set. A lower and constant uncertainty characterizes the downscaled dataset (0.2), while the value of sigma in the IDW data set is proportional to the distance to the closest stations, and the base coefficient is set as 0.4

The values below can be adjusted if the project needs a different balance between the downscaled field and the station based IDW field.

## 1. Import libraries

Run this cell first. It loads the Python libraries used for reading NetCDF files, working with tables, computing distances from stations and exporting GeoTIFF files.


In [1]:
from pathlib import Path
import gc
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")


## 2. User settings

Edit only this cell for a standard run.

Use `comp = "u"` for the eastward wind component and `comp = "v"` for the northward wind component. The notebook expects the IDW file and the downscaled file to use the same month, country and area name.


In [2]:
# Month to process, from 1 to 12.
month = 1

# Wind component to process.
# Use "u" for the eastward component and "v" for the northward component.
comp = "v"

# Country or region name used in the file names.
country = "chile"

# Area name used in the file names.
# Typical values are "continental" or "islands".
area_name = "continental"


## 3. Folder configuration

The paths below are intentionally relative and anonymous, so the notebook can be shared without exposing local server folders.

Before running the notebook, place the input files in the expected folders or modify the paths to match your project structure.


In [3]:
# Base project folder.
# By default, this points to a local "data" folder next to the notebook.
BASE_DIR = Path("../data")

# Folder containing the station table.
STATIONS_DIR = BASE_DIR / "stations" / country

# Folder containing the downscaled NetCDF file.
DOWNSCALED_DIR = BASE_DIR / "downscaled_data" / country / "sub_areas"

# Folder containing the IDW NetCDF file.
IDW_DIR = BASE_DIR / "stations" / country / "idw"

# Folder where the final integrated NetCDF and GeoTIFF will be written.
OUTPUT_DIR = BASE_DIR / "atlas_data" / country / "sub_areas"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Expected input files.
stations_file = STATIONS_DIR / f"allstats_wind_speed_{country}.csv"
downscaled_file = DOWNSCALED_DIR / f"{comp}10m_component_downscaled_{country}_m{month}_{area_name}.nc"
idw_file = IDW_DIR / f"{comp}_idw_{country}_m{month}_{area_name}.nc"

print("Stations file:", stations_file)
print("Downscaled file:", downscaled_file)
print("IDW file:", idw_file)
print("Output folder:", OUTPUT_DIR)


Stations file: ../data/stations/chile/allstats_wind_speed_chile.csv
Downscaled file: ../data/downscaled_data/chile/sub_areas/v10m_component_downscaled_chile_m1_continental.nc
IDW file: ../data/stations/chile/idw/v_idw_chile_m1_continental.nc
Output folder: ../data/atlas_data/chile/sub_areas


## 4. Helper functions

These functions keep the workflow readable. They check that files exist, open datasets, ensure that the spatial reference is EPSG:4326 and save outputs safely.


In [4]:
def check_file_exists(path: Path, label: str) -> None:
    """Stop the notebook with a clear message if an input file is missing."""
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")


def ensure_xarray_epsg4326(xdf):
    """Ensure that an xarray object has longitude/latitude spatial dimensions and EPSG:4326 CRS."""
    xdf = xdf.rio.set_spatial_dims(
        x_dim="longitude",
        y_dim="latitude",
        inplace=False,
    )

    if xdf.rio.crs is None:
        xdf = xdf.rio.write_crs("EPSG:4326", inplace=False)

    return xdf


def save_xarray_netcdf_fast(ds: xr.Dataset, output_path: Path) -> None:
    """Save a NetCDF file using light compression."""
    encoding = {
        var: {"zlib": True, "complevel": 1}
        for var in ds.data_vars
    }

    ds.to_netcdf(
        output_path,
        engine="netcdf4",
        encoding=encoding,
    )


def save_integrated_geotiff(ds: xr.Dataset, output_path: Path) -> None:
    """Export the integrated component as a compressed GeoTIFF."""
    variable_name = f"{comp}10_integrated"

    da = ds[variable_name].astype("float32")
    da = da.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
    da = da.rio.write_crs("EPSG:4326", inplace=False)

    da.rio.to_raster(
        output_path,
        driver="GTiff",
        dtype="float32",
        compress="DEFLATE",
        predictor=3,
        zlevel=9,
        tiled=True,
        BIGTIFF="IF_SAFER",
    )


## 5. Open the input datasets

This cell opens the downscaled product and the IDW product. The latitude axis is sorted to avoid alignment problems during the integration.


In [5]:
def open_input_datasets():
    """Open and prepare the downscaled and IDW datasets."""
    check_file_exists(stations_file, "station CSV file")
    check_file_exists(downscaled_file, "downscaled NetCDF file")
    check_file_exists(idw_file, "IDW NetCDF file")

    downscaled_ds = xr.open_dataset(downscaled_file).sortby("latitude")
    idw_ds = xr.open_dataset(idw_file).sortby("latitude")

    # Some downstream tools expect latitude to be in descending order.
    # This line keeps the behavior of the original workflow.
    downscaled_ds = downscaled_ds.reindex(latitude=downscaled_ds.latitude[::-1])

    downscaled_ds = ensure_xarray_epsg4326(downscaled_ds)
    idw_ds = ensure_xarray_epsg4326(idw_ds)

    return downscaled_ds, idw_ds


downscaled_ds, idw_ds = open_input_datasets()

print(downscaled_ds)
print(idw_ds)


<xarray.Dataset> Size: 4GB
Dimensions:         (latitude: 46848, longitude: 10554)
Coordinates:
  * latitude        (latitude) float32 187kB -17.5 -17.5 -17.5 ... -56.54 -56.54
  * longitude       (longitude) float32 42kB -75.72 -75.72 ... -66.93 -66.93
    spatial_ref     int64 8B 0
Data variables:
    v10_downscaled  (latitude, longitude) float64 4GB ...
<xarray.Dataset> Size: 2GB
Dimensions:      (latitude: 46848, longitude: 10554)
Coordinates:
  * latitude     (latitude) float32 187kB -56.54 -56.54 -56.54 ... -17.5 -17.5
  * longitude    (longitude) float32 42kB -75.72 -75.72 -75.72 ... -66.93 -66.93
    spatial_ref  int64 8B 0
Data variables:
    v_reshaped   (latitude, longitude) float32 2GB ...


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


## 6. Compute the distance based IDW uncertainty

The IDW uncertainty increases with the distance from the nearest weather station. Close to stations, the IDW layer is trusted more. Far from stations, the downscaled reanalysis is trusted more.


In [6]:
def compute_normalized_min_distance(idw_ds: xr.Dataset, stations_file: Path, chunk_size: int = 200_000):
    """Compute the normalized distance from each grid cell to the nearest station."""
    stations_data = pd.read_csv(stations_file)

    required_columns = {"latitude", "longitude"}
    missing_columns = required_columns.difference(stations_data.columns)
    if missing_columns:
        raise ValueError(f"The station CSV is missing these columns: {sorted(missing_columns)}")

    stations = (
        stations_data[["latitude", "longitude"]]
        .drop_duplicates()
        .dropna()
        .to_numpy(dtype=np.float32)
    )

    if len(stations) == 0:
        raise ValueError("No valid station coordinates found in the station CSV file.")

    tree = cKDTree(stations)

    lat_values = idw_ds.latitude.to_numpy().astype(np.float32)
    lon_values = idw_ds.longitude.to_numpy().astype(np.float32)

    lat2d, lon2d = np.meshgrid(lat_values, lon_values, indexing="ij")
    grid_points = np.column_stack([lat2d.ravel(), lon2d.ravel()]).astype(np.float32)

    min_dist = np.empty(grid_points.shape[0], dtype=np.float32)

    for start in range(0, grid_points.shape[0], chunk_size):
        end = min(start + chunk_size, grid_points.shape[0])
        dist, _ = tree.query(grid_points[start:end], k=1)
        min_dist[start:end] = dist.astype(np.float32) ** 3

    min_dist = min_dist.reshape(len(lat_values), len(lon_values))

    min_dist_da = xr.DataArray(
        min_dist,
        coords={"latitude": lat_values, "longitude": lon_values},
        dims=("latitude", "longitude"),
        name="normalized_distance_to_station",
    )

    distance_range = min_dist_da.max() - min_dist_da.min()
    if float(distance_range) == 0:
        return xr.zeros_like(min_dist_da)

    return (min_dist_da - min_dist_da.min()) / distance_range


min_dist_norm = compute_normalized_min_distance(idw_ds, stations_file)
min_dist_norm


<xarray.DataArray 'normalized_distance_to_station' (latitude: 46848,
                                                    longitude: 10554)> Size: 2GB
array([[1.        , 0.9997649 , 0.999528  , ..., 0.02692537, 0.02694047,
        0.0269556 ],
       [0.99964166, 0.99940693, 0.99917006, ..., 0.02688999, 0.02690508,
        0.0269202 ],
       [0.99928224, 0.99904734, 0.9988105 , ..., 0.02685448, 0.02686957,
        0.02688469],
       ...,
       [0.81955093, 0.81918067, 0.81880695, ..., 0.08376478, 0.08384676,
        0.08392878],
       [0.8196101 , 0.81923956, 0.8188659 , ..., 0.08376786, 0.08384984,
        0.08393186],
       [0.819669  , 0.8192986 , 0.81892496, ..., 0.08377097, 0.08385295,
        0.08393496]], dtype=float32)
Coordinates:
  * latitude   (latitude) float32 187kB -56.54 -56.54 -56.54 ... -17.5 -17.5
  * longitude  (longitude) float32 42kB -75.72 -75.72 -75.72 ... -66.93 -66.93

## 7. Define the uncertainty model

In [7]:
def compute_uncertainties(min_dist_norm):
    """Return normalized uncertainty fields for the downscaled and IDW products."""
    # Constant uncertainty assigned to the downscaled product.
    sigma2_downscaled = 0.2 ** 2

    # IDW uncertainty increases with distance from the nearest station.
    sigma2_idw = (0.4 + min_dist_norm) ** 2

    return sigma2_downscaled, sigma2_idw


sigma2_downscaled, sigma2_idw = compute_uncertainties(min_dist_norm)


## 8. Integrate downscaled and IDW wind components

This cell creates the final integrated wind component and saves it both as NetCDF and GeoTIFF.


In [8]:
def apply_scaling(downscaled_ds, sigma2_downscaled, idw_ds, sigma2_idw):
    """Combine downscaled and IDW wind components using uncertainty based weights."""
    downscaled_var = f"{comp}10_downscaled"
    idw_var = f"{comp}_reshaped"
    output_var = f"{comp}10_integrated"

    if downscaled_var not in downscaled_ds:
        raise KeyError(f"Variable '{downscaled_var}' not found in downscaled dataset.")
    if idw_var not in idw_ds:
        raise KeyError(f"Variable '{idw_var}' not found in IDW dataset.")

    # Align datasets on the same latitude and longitude grid.
    downscaled_component, idw_component, sigma2_idw_aligned = xr.align(
        downscaled_ds[downscaled_var],
        idw_ds[idw_var],
        sigma2_idw,
        join="inner",
    )

    if downscaled_component.size == 0:
        raise ValueError("The downscaled and IDW datasets do not overlap on the same grid.")

    integrated = (
        (downscaled_component / sigma2_downscaled) +
        (idw_component / sigma2_idw_aligned)
    ) / (
        (1 / sigma2_downscaled) +
        (1 / sigma2_idw_aligned)
    )

    integrated_ds = integrated.astype("float32").to_dataset(name=output_var)
    integrated_ds = ensure_xarray_epsg4326(integrated_ds)

    netcdf_output = OUTPUT_DIR / f"{output_var}_{country}_m{month}_{area_name}.nc"
    geotiff_output = OUTPUT_DIR / f"{output_var}_{country}_m{month}_{area_name}.tif"

    print("Saving NetCDF:", netcdf_output)
    save_xarray_netcdf_fast(integrated_ds, netcdf_output)

    print("Saving GeoTIFF:", geotiff_output)
    save_integrated_geotiff(integrated_ds, geotiff_output)

    return integrated_ds, netcdf_output, geotiff_output


integrated_ds, netcdf_output, geotiff_output = apply_scaling(
    downscaled_ds,
    sigma2_downscaled,
    idw_ds,
    sigma2_idw,
)

integrated_ds


Saving NetCDF: ../data/atlas_data/chile/sub_areas/v10_integrated_chile_m1_continental.nc
Saving GeoTIFF: ../data/atlas_data/chile/sub_areas/v10_integrated_chile_m1_continental.tif


<xarray.Dataset> Size: 2GB
Dimensions:         (latitude: 46848, longitude: 10554)
Coordinates:
  * latitude        (latitude) float32 187kB -17.5 -17.5 -17.5 ... -56.54 -56.54
  * longitude       (longitude) float32 42kB -75.72 -75.72 ... -66.93 -66.93
    spatial_ref     int64 8B 0
Data variables:
    v10_integrated  (latitude, longitude) float32 2GB nan nan nan ... nan nan

## 9. Optional quick check

This final cell prints the output paths and basic information about the integrated dataset.


In [9]:
print("Integrated dataset saved successfully.")
print("NetCDF:", netcdf_output)
print("GeoTIFF:", geotiff_output)
print(integrated_ds)

gc.collect()


Integrated dataset saved successfully.
NetCDF: ../data/atlas_data/chile/sub_areas/v10_integrated_chile_m1_continental.nc
GeoTIFF: ../data/atlas_data/chile/sub_areas/v10_integrated_chile_m1_continental.tif
<xarray.Dataset> Size: 2GB
Dimensions:         (latitude: 46848, longitude: 10554)
Coordinates:
  * latitude        (latitude) float32 187kB -17.5 -17.5 -17.5 ... -56.54 -56.54
  * longitude       (longitude) float32 42kB -75.72 -75.72 ... -66.93 -66.93
    spatial_ref     int64 8B 0
Data variables:
    v10_integrated  (latitude, longitude) float32 2GB nan nan nan ... nan nan


25